In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Insurance Claim Prediction") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.1.1


In [2]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("hdfs://namenode:8020/data/insurance/raw/AutoInsuranceClaim.csv")

print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 595212
Columns: 59


In [3]:
df.show(10, truncate=False)

+---+------+---------+-------------+---------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+---------+---------+-------------+-------------+-------------+---------+---------+------------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+---------+-------------------+-----------+-------------------+------------------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+--------------+--------------+--------------+--------------+--------------+--------------+
|id |target|ps_ind_01|ps_ind_02_cat|ps_ind_03|ps_ind_04_cat|ps_ind_05_cat|ps_ind_06_bin|ps_ind_07_bin|ps_ind_08_bin|ps_ind_09_bin|ps_ind_10_bin|ps_ind_11_bin|ps_ind_12_bin|ps_ind_13_bin|ps_ind_14|ps_ind_15|ps_ind_16_bin|ps_ind_17_bin

In [4]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- target: integer (nullable = true)
 |-- ps_ind_01: integer (nullable = true)
 |-- ps_ind_02_cat: double (nullable = true)
 |-- ps_ind_03: integer (nullable = true)
 |-- ps_ind_04_cat: double (nullable = true)
 |-- ps_ind_05_cat: double (nullable = true)
 |-- ps_ind_06_bin: integer (nullable = true)
 |-- ps_ind_07_bin: integer (nullable = true)
 |-- ps_ind_08_bin: integer (nullable = true)
 |-- ps_ind_09_bin: integer (nullable = true)
 |-- ps_ind_10_bin: integer (nullable = true)
 |-- ps_ind_11_bin: integer (nullable = true)
 |-- ps_ind_12_bin: integer (nullable = true)
 |-- ps_ind_13_bin: integer (nullable = true)
 |-- ps_ind_14: integer (nullable = true)
 |-- ps_ind_15: integer (nullable = true)
 |-- ps_ind_16_bin: integer (nullable = true)
 |-- ps_ind_17_bin: integer (nullable = true)
 |-- ps_ind_18_bin: integer (nullable = true)
 |-- ps_reg_01: double (nullable = true)
 |-- ps_reg_02: double (nullable = true)
 |-- ps_reg_03: double (nullab

In [5]:
df.groupBy("target").count().show()

+------+------+
|target| count|
+------+------+
|     1| 21694|
|     0|573518|
+------+------+



In [6]:
for field in df.schema.fields:
    print(field.name, field.dataType.simpleString())

id int
target int
ps_ind_01 int
ps_ind_02_cat double
ps_ind_03 int
ps_ind_04_cat double
ps_ind_05_cat double
ps_ind_06_bin int
ps_ind_07_bin int
ps_ind_08_bin int
ps_ind_09_bin int
ps_ind_10_bin int
ps_ind_11_bin int
ps_ind_12_bin int
ps_ind_13_bin int
ps_ind_14 int
ps_ind_15 int
ps_ind_16_bin int
ps_ind_17_bin int
ps_ind_18_bin int
ps_reg_01 double
ps_reg_02 double
ps_reg_03 double
ps_car_01_cat double
ps_car_02_cat int
ps_car_03_cat double
ps_car_04_cat int
ps_car_05_cat double
ps_car_06_cat int
ps_car_07_cat double
ps_car_08_cat int
ps_car_09_cat double
ps_car_10_cat int
ps_car_11_cat int
ps_car_11 double
ps_car_12 double
ps_car_13 double
ps_car_14 double
ps_car_15 double
ps_calc_01 double
ps_calc_02 double
ps_calc_03 double
ps_calc_04 int
ps_calc_05 int
ps_calc_06 int
ps_calc_07 int
ps_calc_08 int
ps_calc_09 int
ps_calc_10 int
ps_calc_11 int
ps_calc_12 int
ps_calc_13 int
ps_calc_14 int
ps_calc_15_bin int
ps_calc_16_bin int
ps_calc_17_bin int
ps_calc_18_bin int
ps_calc_19_bin int
ps

In [7]:
for field in df.schema.fields:
    print(field.name, field.dataType.simpleString())

id int
target int
ps_ind_01 int
ps_ind_02_cat double
ps_ind_03 int
ps_ind_04_cat double
ps_ind_05_cat double
ps_ind_06_bin int
ps_ind_07_bin int
ps_ind_08_bin int
ps_ind_09_bin int
ps_ind_10_bin int
ps_ind_11_bin int
ps_ind_12_bin int
ps_ind_13_bin int
ps_ind_14 int
ps_ind_15 int
ps_ind_16_bin int
ps_ind_17_bin int
ps_ind_18_bin int
ps_reg_01 double
ps_reg_02 double
ps_reg_03 double
ps_car_01_cat double
ps_car_02_cat int
ps_car_03_cat double
ps_car_04_cat int
ps_car_05_cat double
ps_car_06_cat int
ps_car_07_cat double
ps_car_08_cat int
ps_car_09_cat double
ps_car_10_cat int
ps_car_11_cat int
ps_car_11 double
ps_car_12 double
ps_car_13 double
ps_car_14 double
ps_car_15 double
ps_calc_01 double
ps_calc_02 double
ps_calc_03 double
ps_calc_04 int
ps_calc_05 int
ps_calc_06 int
ps_calc_07 int
ps_calc_08 int
ps_calc_09 int
ps_calc_10 int
ps_calc_11 int
ps_calc_12 int
ps_calc_13 int
ps_calc_14 int
ps_calc_15_bin int
ps_calc_16_bin int
ps_calc_17_bin int
ps_calc_18_bin int
ps_calc_19_bin int
ps

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan

In [9]:
spark = SparkSession.builder \
    .appName("Insurance Claim Prediction") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

In [10]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("hdfs://namenode:8020/data/insurance/raw/AutoInsuranceClaim.csv")

In [11]:
print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

Rows: 595212
Columns: 59
root
 |-- id: integer (nullable = true)
 |-- target: integer (nullable = true)
 |-- ps_ind_01: integer (nullable = true)
 |-- ps_ind_02_cat: double (nullable = true)
 |-- ps_ind_03: integer (nullable = true)
 |-- ps_ind_04_cat: double (nullable = true)
 |-- ps_ind_05_cat: double (nullable = true)
 |-- ps_ind_06_bin: integer (nullable = true)
 |-- ps_ind_07_bin: integer (nullable = true)
 |-- ps_ind_08_bin: integer (nullable = true)
 |-- ps_ind_09_bin: integer (nullable = true)
 |-- ps_ind_10_bin: integer (nullable = true)
 |-- ps_ind_11_bin: integer (nullable = true)
 |-- ps_ind_12_bin: integer (nullable = true)
 |-- ps_ind_13_bin: integer (nullable = true)
 |-- ps_ind_14: integer (nullable = true)
 |-- ps_ind_15: integer (nullable = true)
 |-- ps_ind_16_bin: integer (nullable = true)
 |-- ps_ind_17_bin: integer (nullable = true)
 |-- ps_ind_18_bin: integer (nullable = true)
 |-- ps_reg_01: double (nullable = true)
 |-- ps_reg_02: double (nullable = true)
 |-- 

In [12]:
missing = df.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    for c in df.columns
])

missing.show(truncate=False)

+---+------+---------+-------------+---------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+---------+---------+-------------+-------------+-------------+---------+---------+---------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+---------+---------+---------+---------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+--------------+--------------+--------------+--------------+--------------+--------------+
|id |target|ps_ind_01|ps_ind_02_cat|ps_ind_03|ps_ind_04_cat|ps_ind_05_cat|ps_ind_06_bin|ps_ind_07_bin|ps_ind_08_bin|ps_ind_09_bin|ps_ind_10_bin|ps_ind_11_bin|ps_ind_12_bin|ps_ind_13_bin|ps_ind_14|ps_ind_15|ps_ind_16_bin|ps_ind_17_bin|ps_ind_18_bin|ps_reg_01|ps_reg_02|ps_re

In [13]:
df_ml = df.drop("id")

In [14]:
df_ml.groupBy("target").count().orderBy("target").show()

+------+------+
|target| count|
+------+------+
|     0|573518|
|     1| 21694|
+------+------+



In [15]:
missing.show(truncate=False)

+---+------+---------+-------------+---------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+---------+---------+-------------+-------------+-------------+---------+---------+---------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+---------+---------+---------+---------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+--------------+--------------+--------------+--------------+--------------+--------------+
|id |target|ps_ind_01|ps_ind_02_cat|ps_ind_03|ps_ind_04_cat|ps_ind_05_cat|ps_ind_06_bin|ps_ind_07_bin|ps_ind_08_bin|ps_ind_09_bin|ps_ind_10_bin|ps_ind_11_bin|ps_ind_12_bin|ps_ind_13_bin|ps_ind_14|ps_ind_15|ps_ind_16_bin|ps_ind_17_bin|ps_ind_18_bin|ps_reg_01|ps_reg_02|ps_re

In [16]:
from pyspark.sql.functions import col, when, count, isnan

df_ml = df.drop("id")

categorical_cols = [
    c for c in df_ml.columns
    if c.endswith("_cat") or c.endswith("_bin")
]

numeric_cols = [
    c for c in df_ml.columns
    if c not in categorical_cols and c != "target"
]

print("Categorical/Binary columns:", len(categorical_cols))
print("Numeric columns:", len(numeric_cols))

print("Categorical/Binary columns:")
print(categorical_cols)

print("Numeric columns:")
print(numeric_cols)

Categorical/Binary columns: 31
Numeric columns: 26
Categorical/Binary columns:
['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_ind_06_bin', 'ps_ind_07_bin', 'ps_ind_08_bin', 'ps_ind_09_bin', 'ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_16_bin', 'ps_ind_17_bin', 'ps_ind_18_bin', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat', 'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_10_cat', 'ps_car_11_cat', 'ps_calc_15_bin', 'ps_calc_16_bin', 'ps_calc_17_bin', 'ps_calc_18_bin', 'ps_calc_19_bin', 'ps_calc_20_bin']
Numeric columns:
['ps_ind_01', 'ps_ind_03', 'ps_ind_14', 'ps_ind_15', 'ps_reg_01', 'ps_reg_02', 'ps_reg_03', 'ps_car_11', 'ps_car_12', 'ps_car_13', 'ps_car_14', 'ps_car_15', 'ps_calc_01', 'ps_calc_02', 'ps_calc_03', 'ps_calc_04', 'ps_calc_05', 'ps_calc_06', 'ps_calc_07', 'ps_calc_08', 'ps_calc_09', 'ps_calc_10', 'ps_calc_11', 'ps_calc_12', 'ps_calc_13', 'ps_calc_14']


In [17]:
from pyspark.sql.functions import col

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 476116
Testing rows: 119096


In [18]:
print("Training target distribution:")
train_df.groupBy("target").count().orderBy("target").show()

print("Testing target distribution:")
test_df.groupBy("target").count().orderBy("target").show()

Training target distribution:
+------+------+
|target| count|
+------+------+
|     0|458783|
|     1| 17333|
+------+------+

Testing target distribution:
+------+------+
|target| count|
+------+------+
|     0|114735|
|     1|  4361|
+------+------+



In [19]:
from pyspark.sql.functions import col, count, desc

categorical_fill = {}

for c in categorical_cols:
    value = train_df.groupBy(c) \
        .count() \
        .orderBy(desc("count")) \
        .first()[0]
    categorical_fill[c] = value

numeric_fill = {}

for c in numeric_cols:
    value = train_df.approxQuantile(c, [0.5], 0.01)[0]
    numeric_fill[c] = value

print("Categorical values for imputation:")
print(categorical_fill)

print("\nNumeric median values for imputation:")
print(numeric_fill)

Categorical values for imputation:
{'ps_ind_02_cat': 1.0, 'ps_ind_04_cat': 0.0, 'ps_ind_05_cat': 0.0, 'ps_ind_06_bin': 0, 'ps_ind_07_bin': 0, 'ps_ind_08_bin': 0, 'ps_ind_09_bin': 0, 'ps_ind_10_bin': 0, 'ps_ind_11_bin': 0, 'ps_ind_12_bin': 0, 'ps_ind_13_bin': 0, 'ps_ind_16_bin': 1, 'ps_ind_17_bin': 0, 'ps_ind_18_bin': 0, 'ps_car_01_cat': 11.0, 'ps_car_02_cat': 1, 'ps_car_03_cat': None, 'ps_car_04_cat': 0, 'ps_car_05_cat': None, 'ps_car_06_cat': 11, 'ps_car_07_cat': 1.0, 'ps_car_08_cat': 1, 'ps_car_09_cat': 2.0, 'ps_car_10_cat': 1, 'ps_car_11_cat': 104, 'ps_calc_15_bin': 0, 'ps_calc_16_bin': 1, 'ps_calc_17_bin': 1, 'ps_calc_18_bin': 0, 'ps_calc_19_bin': 0, 'ps_calc_20_bin': 0}

Numeric median values for imputation:
{'ps_ind_01': 1.0, 'ps_ind_03': 4.0, 'ps_ind_14': 0.0, 'ps_ind_15': 7.0, 'ps_reg_01': 0.7, 'ps_reg_02': 0.3, 'ps_reg_03': 0.799218368, 'ps_car_11': 3.0, 'ps_car_12': 0.374165739, 'ps_car_13': 0.76469083, 'ps_car_14': 0.371483512, 'ps_car_15': 3.31662479, 'ps_calc_01': 0.4, 'ps

In [20]:
print("Categorical columns with None:")
print([c for c, v in categorical_fill.items() if v is None])

print("Numeric columns with None:")
print([c for c, v in numeric_fill.items() if v is None])

Categorical columns with None:
['ps_car_03_cat', 'ps_car_05_cat']
Numeric columns with None:
[]


In [21]:
from pyspark.sql.functions import desc

categorical_fill = {}

for c in categorical_cols:
    row = train_df.where(col(c).isNotNull()) \
        .groupBy(c) \
        .count() \
        .orderBy(desc("count")) \
        .first()

    if row is not None:
        categorical_fill[c] = row[0]

numeric_fill = {}

for c in numeric_cols:
    values = train_df.approxQuantile(c, [0.5], 0.01)
    
    if values and values[0] is not None:
        numeric_fill[c] = values[0]

print("Categorical fill values:", categorical_fill)
print("Numeric fill values:", numeric_fill)

Categorical fill values: {'ps_ind_02_cat': 1.0, 'ps_ind_04_cat': 0.0, 'ps_ind_05_cat': 0.0, 'ps_ind_06_bin': 0, 'ps_ind_07_bin': 0, 'ps_ind_08_bin': 0, 'ps_ind_09_bin': 0, 'ps_ind_10_bin': 0, 'ps_ind_11_bin': 0, 'ps_ind_12_bin': 0, 'ps_ind_13_bin': 0, 'ps_ind_16_bin': 1, 'ps_ind_17_bin': 0, 'ps_ind_18_bin': 0, 'ps_car_01_cat': 11.0, 'ps_car_02_cat': 1, 'ps_car_03_cat': 1.0, 'ps_car_04_cat': 0, 'ps_car_05_cat': 1.0, 'ps_car_06_cat': 11, 'ps_car_07_cat': 1.0, 'ps_car_08_cat': 1, 'ps_car_09_cat': 2.0, 'ps_car_10_cat': 1, 'ps_car_11_cat': 104, 'ps_calc_15_bin': 0, 'ps_calc_16_bin': 1, 'ps_calc_17_bin': 1, 'ps_calc_18_bin': 0, 'ps_calc_19_bin': 0, 'ps_calc_20_bin': 0}
Numeric fill values: {'ps_ind_01': 1.0, 'ps_ind_03': 4.0, 'ps_ind_14': 0.0, 'ps_ind_15': 7.0, 'ps_reg_01': 0.7, 'ps_reg_02': 0.3, 'ps_reg_03': 0.799218368, 'ps_car_11': 3.0, 'ps_car_12': 0.374165739, 'ps_car_13': 0.76469083, 'ps_car_14': 0.371483512, 'ps_car_15': 3.31662479, 'ps_calc_01': 0.4, 'ps_calc_02': 0.4, 'ps_calc_03': 

In [22]:
train_clean = train_df.fillna(categorical_fill)
train_clean = train_clean.fillna(numeric_fill)

test_clean = test_df.fillna(categorical_fill)
test_clean = test_clean.fillna(numeric_fill)

print("Training rows:", train_clean.count())
print("Testing rows:", test_clean.count())

Training rows: 476116
Testing rows: 119096


In [23]:
from pyspark.sql.functions import when, count, col, isnan

missing_train = train_clean.select([
    count(
        when(col(c).isNull() | isnan(col(c)), c)
    ).alias(c)
    for c in train_clean.columns
])

missing_train.show(truncate=False)

+------+---------+-------------+---------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+---------+---------+-------------+-------------+-------------+---------+---------+---------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+---------+---------+---------+---------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+--------------+--------------+--------------+--------------+--------------+--------------+
|target|ps_ind_01|ps_ind_02_cat|ps_ind_03|ps_ind_04_cat|ps_ind_05_cat|ps_ind_06_bin|ps_ind_07_bin|ps_ind_08_bin|ps_ind_09_bin|ps_ind_10_bin|ps_ind_11_bin|ps_ind_12_bin|ps_ind_13_bin|ps_ind_14|ps_ind_15|ps_ind_16_bin|ps_ind_17_bin|ps_ind_18_bin|ps_reg_01|ps_reg_02|ps_reg_03|ps_

In [24]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [c for c in train_clean.columns if c != "target"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_vector = assembler.transform(train_clean)
test_vector = assembler.transform(test_clean)

train_vector.select("target", "features").show(5, truncate=False)

+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|target|features                                                                                                                                                                                                                                                                                                              |
+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0     |(57,[1,8,11,13,14,15,20,21,22,23

In [25]:
print("Number of features:", len(feature_cols))
print("Training rows:", train_vector.count())
print("Testing rows:", test_vector.count())

Number of features: 57
Training rows: 476116
Testing rows: 119096


In [26]:
from pyspark.sql.functions import col, when

total = train_vector.count()
positive = train_vector.filter(col("target") == 1).count()
negative = train_vector.filter(col("target") == 0).count()

weight_for_1 = negative / positive

print("Total training records:", total)
print("Non-claim records:", negative)
print("Claim records:", positive)
print("Weight for claim class:", weight_for_1)

train_weighted = train_vector.withColumn(
    "weight",
    when(col("target") == 1, weight_for_1).otherwise(1.0)
)

test_weighted = test_vector.withColumn(
    "weight",
    when(col("target") == 1, weight_for_1).otherwise(1.0)
)

train_weighted.groupBy("target").agg(
    {"weight": "avg"}
).show()

Total training records: 476116
Non-claim records: 458783
Claim records: 17333
Weight for claim class: 26.468759014596433
+------+------------------+
|target|       avg(weight)|
+------+------------------+
|     1|26.468759014597033|
|     0|               1.0|
+------+------------------+



In [27]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="target",
    weightCol="weight",
    maxIter=50
)

lr_model = lr.fit(train_weighted)

lr_predictions = lr_model.transform(test_weighted)

print("Logistic Regression completed successfully")

lr_predictions.select(
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

Logistic Regression completed successfully
+------+----------+----------------------------------------+
|target|prediction|probability                             |
+------+----------+----------------------------------------+
|0     |1.0       |[0.48810347469088194,0.5118965253091181]|
|0     |0.0       |[0.6264608583766406,0.37353914162335944]|
|0     |0.0       |[0.5906357524089284,0.4093642475910716] |
|0     |0.0       |[0.6014336727905456,0.3985663272094544] |
|0     |0.0       |[0.531184738755858,0.46881526124414197] |
|0     |0.0       |[0.5594448204813423,0.44055517951865775]|
|0     |0.0       |[0.6342268294133893,0.36577317058661074]|
|0     |0.0       |[0.5751688735767163,0.4248311264232837] |
|0     |0.0       |[0.6134245400216066,0.3865754599783934] |
|0     |0.0       |[0.5937645520259242,0.40623544797407585]|
+------+----------+----------------------------------------+
only showing top 10 rows



In [28]:
lr_predictions.select(
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

+------+----------+----------------------------------------+
|target|prediction|probability                             |
+------+----------+----------------------------------------+
|0     |1.0       |[0.48810347469088194,0.5118965253091181]|
|0     |0.0       |[0.6264608583766406,0.37353914162335944]|
|0     |0.0       |[0.5906357524089284,0.4093642475910716] |
|0     |0.0       |[0.6014336727905456,0.3985663272094544] |
|0     |0.0       |[0.531184738755858,0.46881526124414197] |
|0     |0.0       |[0.5594448204813423,0.44055517951865775]|
|0     |0.0       |[0.6342268294133893,0.36577317058661074]|
|0     |0.0       |[0.5751688735767163,0.4248311264232837] |
|0     |0.0       |[0.6134245400216066,0.3865754599783934] |
|0     |0.0       |[0.5937645520259242,0.40623544797407585]|
+------+----------+----------------------------------------+
only showing top 10 rows



In [29]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(lr_predictions)

precision = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(lr_predictions)

recall = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(lr_predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="f1"
).evaluate(lr_predictions)

roc_auc = BinaryClassificationEvaluator(
    labelCol="target",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(lr_predictions)

print("Logistic Regression Results")
print("----------------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

Logistic Regression Results
----------------------------
Accuracy : 0.6235
Precision: 0.9397
Recall   : 0.6235
F1 Score : 0.7378
ROC-AUC  : 0.6277


In [30]:
lr_predictions.groupBy(
    "target",
    "prediction"
).count().orderBy(
    "target",
    "prediction"
).show()

+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     0|       0.0|71861|
|     0|       1.0|42874|
|     1|       0.0| 1965|
|     1|       1.0| 2396|
+------+----------+-----+



In [31]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="target",
    weightCol="weight",
    maxDepth=8,
    seed=42
)

dt_model = dt.fit(train_weighted)

dt_predictions = dt_model.transform(test_weighted)

print("Decision Tree completed successfully")

dt_predictions.select(
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

Decision Tree completed successfully
+------+----------+----------------------------------------+
|target|prediction|probability                             |
+------+----------+----------------------------------------+
|0     |0.0       |[0.5271375604935575,0.4728624395064424] |
|0     |1.0       |[0.48229178946226614,0.5177082105377337]|
|0     |0.0       |[0.5633528235892118,0.43664717641078815]|
|0     |0.0       |[0.5633528235892118,0.43664717641078815]|
|0     |0.0       |[1.0,0.0]                               |
|0     |0.0       |[0.5983983361846652,0.40160166381533485]|
|0     |0.0       |[0.6609236392960584,0.3390763607039416] |
|0     |1.0       |[0.3418802652515899,0.6581197347484101] |
|0     |0.0       |[0.6609236392960584,0.3390763607039416] |
|0     |0.0       |[1.0,0.0]                               |
+------+----------+----------------------------------------+
only showing top 10 rows



In [32]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

dt_accuracy = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(dt_predictions)

dt_precision = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(dt_predictions)

dt_recall = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(dt_predictions)

dt_f1 = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="f1"
).evaluate(dt_predictions)

dt_roc_auc = BinaryClassificationEvaluator(
    labelCol="target",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(dt_predictions)

print("Decision Tree Results")
print("---------------------")
print("Accuracy :", round(dt_accuracy, 4))
print("Precision:", round(dt_precision, 4))
print("Recall   :", round(dt_recall, 4))
print("F1 Score :", round(dt_f1, 4))
print("ROC-AUC  :", round(dt_roc_auc, 4))

Decision Tree Results
---------------------
Accuracy : 0.6946
Precision: 0.9373
Recall   : 0.6946
F1 Score : 0.7899
ROC-AUC  : 0.4515


In [33]:
dt_predictions.groupBy(
    "target",
    "prediction"
).count().orderBy(
    "target",
    "prediction"
).show()

+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     0|       0.0|80796|
|     0|       1.0|33939|
|     1|       0.0| 2427|
|     1|       1.0| 1934|
+------+----------+-----+



In [34]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="target",
    weightCol="weight",
    numTrees=50,
    maxDepth=8,
    seed=42
)

rf_model = rf.fit(train_weighted)

rf_predictions = rf_model.transform(test_weighted)

print("Random Forest completed successfully")

rf_predictions.select(
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

Random Forest completed successfully
+------+----------+----------------------------------------+
|target|prediction|probability                             |
+------+----------+----------------------------------------+
|0     |1.0       |[0.4927952694535971,0.5072047305464029] |
|0     |0.0       |[0.574753445101641,0.42524655489835905] |
|0     |0.0       |[0.5817192706158669,0.41828072938413313]|
|0     |0.0       |[0.5412760262237105,0.4587239737762896] |
|0     |1.0       |[0.4920027301825822,0.5079972698174179] |
|0     |0.0       |[0.5024940591281879,0.497505940871812]  |
|0     |0.0       |[0.5495887592414527,0.4504112407585473] |
|0     |1.0       |[0.44605689971866996,0.5539431002813301]|
|0     |0.0       |[0.6247428348412867,0.3752571651587133] |
|0     |1.0       |[0.4952917514894733,0.5047082485105266] |
+------+----------+----------------------------------------+
only showing top 10 rows



In [35]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

rf_accuracy = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(rf_predictions)

rf_precision = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(rf_predictions)

rf_recall = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(rf_predictions)

rf_f1 = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="f1"
).evaluate(rf_predictions)

rf_roc_auc = BinaryClassificationEvaluator(
    labelCol="target",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(rf_predictions)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))
print("ROC-AUC  :", round(rf_roc_auc, 4))

Random Forest Results
---------------------
Accuracy : 0.6479
Precision: 0.9394
Recall   : 0.6479
F1 Score : 0.7562
ROC-AUC  : 0.6255


In [36]:
from pyspark.sql.functions import col

def claim_metrics(predictions, model_name):
    tp = predictions.filter(
        (col("target") == 1) & (col("prediction") == 1)
    ).count()

    fp = predictions.filter(
        (col("target") == 0) & (col("prediction") == 1)
    ).count()

    fn = predictions.filter(
        (col("target") == 1) & (col("prediction") == 0)
    ).count()

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)

    print(model_name)
    print("--------------------")
    print("TP:", tp)
    print("FP:", fp)
    print("FN:", fn)
    print("Claim Precision:", round(precision, 4))
    print("Claim Recall   :", round(recall, 4))
    print("Claim F1       :", round(f1, 4))
    print()

claim_metrics(lr_predictions, "Logistic Regression")
claim_metrics(dt_predictions, "Decision Tree")
claim_metrics(rf_predictions, "Random Forest")

Logistic Regression
--------------------
TP: 2396
FP: 42874
FN: 1965
Claim Precision: 0.0529
Claim Recall   : 0.5494
Claim F1       : 0.0966

Decision Tree
--------------------
TP: 1934
FP: 33939
FN: 2427
Claim Precision: 0.0539
Claim Recall   : 0.4435
Claim F1       : 0.0961

Random Forest
--------------------
TP: 2287
FP: 39858
FN: 2074
Claim Precision: 0.0543
Claim Recall   : 0.5244
Claim F1       : 0.0984



In [37]:
rf_results = rf_predictions.select(
    "target",
    "prediction"
)

rf_results.show(10)

rf_results.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("hdfs://namenode:8020/data/insurance/results/random_forest_predictions")

print("Random Forest predictions saved to HDFS successfully")

+------+----------+
|target|prediction|
+------+----------+
|     0|       1.0|
|     0|       0.0|
|     0|       0.0|
|     0|       0.0|
|     0|       1.0|
|     0|       0.0|
|     0|       0.0|
|     0|       1.0|
|     0|       0.0|
|     0|       1.0|
+------+----------+
only showing top 10 rows

Random Forest predictions saved to HDFS successfully


In [38]:
comparison_data = [
    ("Logistic Regression", 0.6200, 0.0526, 0.5504, 0.0960, 0.6205),
    ("Decision Tree", 0.6775, 0.0526, 0.4579, 0.0943, 0.4390),
    ("Random Forest", 0.6453, 0.0537, 0.5217, 0.0974, 0.6213)
]

import pandas as pd

comparison_pd = pd.DataFrame(
    comparison_data,
    columns=[
        "model",
        "accuracy",
        "claim_precision",
        "claim_recall",
        "claim_f1",
        "roc_auc"
    ]
)

comparison_pd

,model,accuracy,claim_precision,claim_recall,claim_f1,roc_auc
0,Logistic Regression,0.6200,0.0526,0.5504,0.0960,0.6205
1,Decision Tree,0.6775,0.0526,0.4579,0.0943,0.4390
2,Random Forest,0.6453,0.0537,0.5217,0.0974,0.6213
